# 特征提取

输出 `features/<model>.npz`（`image_ids`、`features`）。

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from fmspace.data import load_metadata
from fmspace.extract import extract, load_features, save_features
from fmspace.models import MODELS, load_encoder

DATA_DIR = Path(r"C:\M1\master paper\DataSet\DeepDRiD")
FEATURE_DIR = Path("../features")
FEATURE_DIR.mkdir(exist_ok=True)

c:\M1\envs\fmspace\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
metadata = load_metadata(DATA_DIR)
metadata

,patient_id,image_id,path,eye,eye_dr_level,overall_quality,clarity,field_definition,artifact
0,1,1_l1,C:\M1\master paper\DataSet\DeepDRiD\regular-fu...,L,0,0,8,8,4
1,1,1_l2,C:\M1\master paper\DataSet\DeepDRiD\regular-fu...,L,0,0,8,8,0
2,1,1_r1,C:\M1\master paper\DataSet\DeepDRiD\regular-fu...,R,0,0,8,8,4
3,1,1_r2,C:\M1\master paper\DataSet\DeepDRiD\regular-fu...,R,0,0,10,8,0
4,2,2_l1,C:\M1\master paper\DataSet\DeepDRiD\regular-fu...,L,2,0,10,8,6
...,...,...,...,...,...,...,...,...,...
1184,329,329_r2,C:\M1\master paper\DataSet\DeepDRiD\regular-fu...,R,1,1,8,10,0
1185,330,330_l1,C:\M1\master paper\DataSet\DeepDRiD\regular-fu...,L,1,0,10,8,0
1186,330,330_l2,C:\M1\master paper\DataSet\DeepDRiD\regular-fu...,L,1,0,10,8,0
1187,330,330_r1,C:\M1\master paper\DataSet\DeepDRiD\regular-fu...,R,0,1,10,8,0


## 试跑

In [6]:
X = extract(load_encoder("flair"), metadata["path"][:8])
X.shape

100%|██████████| 1/1 [00:01<00:00,  1.71s/it]


(8, 512)

## 全量提取

In [7]:
for name in MODELS:
    X = extract(load_encoder(name), metadata["path"])
    save_features(FEATURE_DIR / f"{name}.npz", metadata["image_id"], X)

100%|██████████| 38/38 [04:03<00:00,  6.41s/it]


## 和旧特征比较

L2 归一化后逐张算余弦相似度。

In [8]:
OLD_DIR = Path(r"C:\M1\SAMIT\project\SAMIT\features\DeepDRiD")


def l2(x):
    return x / np.linalg.norm(x, axis=1, keepdims=True)


cos = pd.DataFrame(
    {
        name: (
            l2(load_features(FEATURE_DIR / f"{name}.npz", metadata["image_id"]))
            * l2(load_features(OLD_DIR / f"{name}.npz", metadata["image_id"]))
        ).sum(axis=1)
        for name in MODELS
    }
)
summary = cos.describe().T[["min", "50%"]]
summary.to_csv("../results/feature_check.csv")
summary

,min,50%
siglip,0.999999,1.0
medsiglip,0.999999,1.0
conceptclip,0.999999,1.0
flair,0.999999,1.0
